|<h2>Course:</h2>|<h1><a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">A deep understanding of AI language model mechanisms</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Large language models<h1>|
|<h2>Section:</h2>|<h1>Build a GPT<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: How many parameters?<b></h1>|

<br>

<h5><b>Teacher:</b> Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h5>
<h5><b>Course URL:</b> <a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">udemy.com/course/dullms_x/?couponCode=202508</a></h5>
<i>Using the code without the course may lead to confusion or errors.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch

!pip install torchinfo
from torchinfo import summary

# svg plots
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

In [ ]:
# libraries for GPT models and their tokenizer
from transformers import AutoModelForCausalLM,GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Exercise 1: Import GPT2 models

In [ ]:
# dictionary of modelname:identifier
model_ids = {
    'small':  'gpt2',        # 124M
    'medium': 'gpt2-medium', # 355M
    'large':  'gpt2-large',  # 774M
    'xl':     'gpt2-xl'      # 1.6B (including this model really slows things down...)
}

# load all models into a dictionary
models = {}
for name, id in model_ids.items():
  models[name] = AutoModelForCausalLM.from_pretrained(id)

In [ ]:
models.keys()

In [ ]:
# now you can iterate over them
for name, model in models.items():
  print(f'{sum(p.numel() for p in model.parameters()):13,} parameters in gpt2-{name}')

# Exercise 2: Compare GPT2-small against "model 5"

In [ ]:
# same data for all exercises
x = tokenizer.encode('This is a test. One that we have done countless times before.',return_tensors='pt')
x

In [ ]:
# summary of model and parameters
sumry = summary(models['small'], input_data=x, col_names=['input_size','output_size','num_params'])
print(sumry)

In [ ]:
finalHeadCount = torch.numel(models['small'].lm_head.weight)

print(f'Total trainable parameters: {sumry.total_params - finalHeadCount:,}')

In [ ]:
# show that token embedding and final output layer are the same
plt.figure(figsize=(8,4))

tokW = models['small'].transformer.wte.weight[0].detach()
head = models['small'].lm_head.weight[0].detach()

plt.plot(tokW,color=[.9,.7,.9],label='Token embeddings')
plt.plot(head,'o',markerfacecolor=[.9,.9,.7],markersize=3,label='Unembeddings')

plt.legend()
plt.gca().set(xlabel='Dimension',ylabel='Value',xlim=[0,len(tokW)],
              title=f'Correlation = {np.corrcoef(tokW,head)[0,1]:.3f}')

plt.show()

# Exercise 3: Weight and bias parameters, as percentage of total



In [ ]:
for name,mat in models['small'].named_parameters():
  print(name)

In [ ]:
# initialize
param_counts = np.zeros((len(models.keys()),3),dtype=int)

for idx,(modelname,model) in enumerate(models.items()):

  # get the summary for all parameters
  sumry = summary(model, input_data=x, col_names=['input_size','output_size','num_params'])
  finalHeadCount = torch.numel(model.lm_head.weight)
  param_counts[idx,2] = sumry.total_params - finalHeadCount

  # loop through all parameters and increment the parameter count
  for layername,mat in model.named_parameters():
    if 'weight' in layername:
      param_counts[idx,0] += torch.numel(mat)
    elif 'bias' in layername:
      param_counts[idx,1] += torch.numel(mat)

  print(f'\n** Model "{modelname}":')
  print(f'Total weights: {param_counts[idx,0]:13,d} ({100*param_counts[idx,0]/param_counts[idx,2]:6.3f}% of all params)')
  print(f'Total biases:  {param_counts[idx,1]:13,d} ({100*param_counts[idx,1]/param_counts[idx,2]:6.3f}% of all params)')

# Exercise 4: Parameters in attention vs. MLP blocks

In [ ]:
# initialize
param_counts = np.zeros((len(models.keys()),3),dtype=int)

for idx,(modelname,model) in enumerate(models.items()):

  # get the summary for all parameters
  sumry = summary(model, input_data=x, col_names=['input_size','output_size','num_params'])
  finalHeadCount = torch.numel(model.lm_head.weight)
  param_counts[idx,2] = sumry.total_params - finalHeadCount

  # loop through all parameters and increment parameter count
  for layername,mat in model.named_parameters():
    if ('attn' in layername) and ('weight' in layername):
      param_counts[idx,0] += torch.numel(mat)
    elif ('mlp' in layername) and ('weight' in layername):
      param_counts[idx,1] += torch.numel(mat)

  print(f'\n** Model "{modelname}" ({len(model.transformer.h)} transformer blocks):')
  print(f'Att weights: {param_counts[idx,0]:11,d} ({100*param_counts[idx,0]/param_counts[idx,2]:5.2f}% of all params)')
  print(f'MLP weights: {param_counts[idx,1]:11,d} ({100*param_counts[idx,1]/param_counts[idx,2]:5.2f}% of all params)')

In [ ]:
plt.bar(np.arange(0,7,2)-.4,100*param_counts[:,0] / param_counts[:,2],label='Attention',color=[.85,.4,.02])
plt.bar(np.arange(0,7,2)+.4,100*param_counts[:,1] / param_counts[:,2],label='MLP',color=[.3,.01,.42])

plt.legend()
plt.gca().set(xticks=range(0,7,2),xticklabels=models.keys(),xlabel='GPT2 model version',
              ylabel='Percent of total model weights',title='Percentage weights per layer type')
plt.show()

# Exercise 5: How many layernorm parameters?

In [ ]:
# initialize
param_counts = np.zeros((len(models.keys()),3),dtype=int)

for idx,(modelname,model) in enumerate(models.items()):

  # get the summary for all parameters
  sumry = summary(model, input_data=x, col_names=['input_size','output_size','num_params'])
  finalHeadCount = torch.numel(model.lm_head.weight)
  param_counts[idx,2] = sumry.total_params - finalHeadCount

  # loop through all parameters and increment parameter count
  for layername,mat in model.named_parameters():
    if ('ln_' in layername) and ('weight' in layername):
      param_counts[idx,0] += torch.numel(mat)
    elif ('ln_' in layername) and ('bias' in layername):
      param_counts[idx,1] += torch.numel(mat)

  print(f'\n** Model "{modelname}" ({len(model.transformer.h)} transformer blocks):')
  print(f'LayerNorm weight: {param_counts[idx,0]:11,d} ({100*param_counts[idx,0]/param_counts[idx,2]:5.4f}% of all params)')
  print(f'LayerNorm biases: {param_counts[idx,1]:11,d} ({100*param_counts[idx,1]/param_counts[idx,2]:5.4f}% of all params)')

In [ ]:
plt.bar(np.arange(0,7,2)-.4,100*param_counts[:,0] / param_counts[:,2],label='Weights',color=[.85,.4,.02])
plt.bar(np.arange(0,7,2)+.4,100*param_counts[:,1] / param_counts[:,2],label='Biases',color=[.3,.01,.42])

plt.legend()
plt.gca().set(xticks=range(0,7,2),xticklabels=models.keys(),xlabel='GPT2 model version',
              ylabel='Percent of total model weights',title='Percentage weights in LayerNorm layers')
plt.show()